# Inference Test — Fine-Tuned Scenario Generator (Colab)

Generates tri-modal scenarios for the **held-out** test inputs using `run_inference.py`. All inference logic lives in that script and uses the **same** `prompt_format.build_chat` as training, so the prompt cannot drift.

It sets eos/pad ids (the model stops) and decodes **only the new tokens** (no echoed prompt). Upload your adapter folder into `./output` first.

In [ ]:
!pip install -q -r requirements_finetune.txt
print('Installed.')

In [ ]:
# Ship the fixed repo modules into this runtime (source of truth).
# run_inference.py + shared modules + held-out inputs.
import pathlib
pathlib.Path("prompt_format.py").write_text("\"\"\"\nprompt_format.py\n----------------\nTHE single source of truth for the prompt that wraps each training record and\neach inference request. Training (finetuning.py) and every inference path\n(run_inference.py, evaluation/evaluate.py, the generated notebooks) all import\n``build_chat`` from here, so the train-time and infer-time prompts can never\ndrift apart. Prompt drift was the original root cause of the off-format model\noutput, so this module is deliberately small and shared.\n\nWhat the prompt looks like for Mistral-7B-Instruct-v0.3\n------------------------------------------------------\nMistral's chat template has no separate ``system`` role that survives a single\nuser turn, so we fold the full ``instruction`` (the tri-modal formatting rules)\ninto the user turn together with the record's ``input``::\n\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST]                 # inference prompt\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST] {output}</s>    # full training text\n\nThe inference prompt is, by construction, a strict prefix of the training text,\nwhich is exactly what the model needs to continue the completion at inference.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\n\n# Mistral-7B-Instruct-v0.3 special tokens. Used only by the no-dependency manual\n# fallback below; when a real tokenizer is supplied we read these from it.\nDEFAULT_BOS = \"<s>\"\nDEFAULT_EOS = \"</s>\"\n\n# The marker that separates the prompt from the completion. Loss is computed only\n# on the tokens that follow this marker (see finetuning.py completion-only setup).\nRESPONSE_MARKER = \"[/INST]\"\n\n\ndef build_messages(record: dict, for_training: bool) -> list[dict]:\n    \"\"\"\n    Build the chat ``messages`` list for one record.\n\n    The full ``instruction`` field (all the [NATURAL LANGUAGE] / [MATHEMATICAL\n    EXPRESSION] / [ATTACK GRAPH] rules) is folded into the user turn — never the\n    short one-liner that older code used.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    user_content = f\"{instruction}\\n\\n{input_text}\"\n\n    messages = [{\"role\": \"user\", \"content\": user_content}]\n    if for_training:\n        output = (record.get(\"output\") or \"\").strip()\n        messages.append({\"role\": \"assistant\", \"content\": output})\n    return messages\n\n\ndef _manual_chat(record: dict, for_training: bool,\n                 bos: str = DEFAULT_BOS, eos: str = DEFAULT_EOS) -> str:\n    \"\"\"\n    Dependency-free fallback that reproduces Mistral's chat template byte-for-byte.\n\n    Mirrors output/chat_template.jinja:\n      user turn      -> \"[INST] \" + content + \"[/INST]\"   (bos prepended once)\n      assistant turn -> \" \" + content.strip() + eos\n    Used when no tokenizer is available (e.g. ``--check_data`` on a machine with\n    no transformers installed). The infer string stays a prefix of the train\n    string here too.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    prompt = f\"{bos}[INST] {instruction}\\n\\n{input_text}[/INST]\"\n    if not for_training:\n        return prompt\n    output = (record.get(\"output\") or \"\").strip()\n    return f\"{prompt} {output}{eos}\"\n\n\ndef build_chat(record: dict, tokenizer: Optional[object], for_training: bool) -> str:\n    \"\"\"\n    Return the canonical prompt string for ``record``.\n\n    Parameters\n    ----------\n    record       : dict with at least ``instruction`` and ``input`` (and\n                   ``output`` when ``for_training`` is True).\n    tokenizer    : a HF tokenizer (uses ``apply_chat_template``) or ``None``\n                   (falls back to the manual template above).\n    for_training : True  -> include the assistant turn + EOS (full training text)\n                   False -> stop after ``[/INST]`` (the generation prompt)\n\n    Guarantee: ``build_chat(rec, tok, False)`` is always a strict prefix of\n    ``build_chat(rec, tok, True)`` for the same record/tokenizer.\n    \"\"\"\n    if tokenizer is not None and hasattr(tokenizer, \"apply_chat_template\"):\n        try:\n            return tokenizer.apply_chat_template(\n                build_messages(record, for_training),\n                tokenize=False,\n                add_generation_prompt=not for_training,\n            )\n        except Exception:\n            # Some odd template/version combo — fall back to the manual template,\n            # using the tokenizer's real special tokens so it stays consistent.\n            bos = getattr(tokenizer, \"bos_token\", None) or DEFAULT_BOS\n            eos = getattr(tokenizer, \"eos_token\", None) or DEFAULT_EOS\n            return _manual_chat(record, for_training, bos, eos)\n    return _manual_chat(record, for_training)\n\n\ndef assert_parity(records: list[dict], tokenizer: Optional[object] = None) -> None:\n    \"\"\"\n    Assert that the inference prompt is a strict prefix of the full training text\n    for every record. This is the train/inference-parity guard (fix A acceptance).\n    Raises AssertionError on the first violation.\n    \"\"\"\n    for i, rec in enumerate(records):\n        infer = build_chat(rec, tokenizer, for_training=False)\n        train = build_chat(rec, tokenizer, for_training=True)\n        assert train.startswith(infer), (\n            f\"Parity violated for record {i} \"\n            f\"(source_id={rec.get('_meta', {}).get('source_id')}): \"\n            \"inference prompt is not a prefix of the training text.\"\n        )\n        assert len(train) > len(infer), (\n            f\"Record {i} produced an empty completion span.\"\n        )\n\n\ndef response_template_ids(tokenizer, sample_text: str, marker: str = RESPONSE_MARKER) -> list[int]:\n    \"\"\"\n    Compute the token ids of ``marker`` (``[/INST]``) **in context**.\n\n    The same string can tokenize differently standalone vs. mid-sequence, which\n    is the classic footgun that makes DataCollatorForCompletionOnlyLM silently\n    mask the entire example. We therefore diff the tokenization of the text up to\n    *and including* the marker against the text up to *just before* it, and use\n    the trailing ids — the ids the marker actually contributes in this context.\n\n    Requires a real tokenizer; only called on the training/smoke path.\n    \"\"\"\n    cut = sample_text.index(marker)\n    end = cut + len(marker)\n    before_ids = tokenizer(sample_text[:cut], add_special_tokens=False)[\"input_ids\"]\n    upto_ids = tokenizer(sample_text[:end], add_special_tokens=False)[\"input_ids\"]\n\n    # The marker's in-context ids are whatever was appended after `before_ids`.\n    # Guard against rare boundary re-tokenization of the last shared token.\n    k = 0\n    while k < len(before_ids) and k < len(upto_ids) and before_ids[k] == upto_ids[k]:\n        k += 1\n    ids = upto_ids[k:]\n    if not ids:\n        # Fallback: standalone encoding (better than an empty template).\n        ids = tokenizer(marker, add_special_tokens=False)[\"input_ids\"]\n    return ids\n", encoding='utf-8')
pathlib.Path("data_split.py").write_text("\"\"\"\ndata_split.py\n-------------\nTHE single source of truth for splitting the dataset into train / validation /\ntest. Persisting the split here (and reading it everywhere) removes the old\ntrain/eval mismatch where finetuning.py did its own random split while\nmake_test_inputs.py treated a different file's tail as \"unseen\" — so the\n\"unseen\" test set was not guaranteed unseen.\n\nHow it works\n------------\n* The split is computed once, deterministically, from a single seed, keyed on\n  each record's stable ``_meta.source_id``.\n* The held-out **test** records are written to ``evaluation/test_inputs.jsonl``.\n* finetuning.py trains on the **complement** of that file (train + val only);\n  make_test_inputs.py creates it; evaluate.py reads it. Everyone shares one\n  artifact, so train ∩ test is provably empty.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Optional\n\n\nDEFAULT_TEST_INPUTS = \"evaluation/test_inputs.jsonl\"\n\n\ndef load_records(jsonl_path: str | Path) -> list[dict]:\n    \"\"\"Read a JSONL dataset, skipping blank / malformed lines.\"\"\"\n    records = []\n    with open(jsonl_path, \"r\", encoding=\"utf-8\") as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                records.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue\n    return records\n\n\ndef source_id(record: dict, idx: int) -> str:\n    \"\"\"Stable id for a record: its _meta.source_id, else a positional fallback.\"\"\"\n    sid = (record.get(\"_meta\") or {}).get(\"source_id\")\n    return sid if sid else f\"idx-{idx}\"\n\n\ndef _annotate(records: list[dict]) -> list[dict]:\n    \"\"\"Attach the original index + canonical source_id to each record (copy).\"\"\"\n    out = []\n    for i, rec in enumerate(records):\n        item = dict(rec)\n        item[\"idx\"] = i\n        item[\"source_id\"] = source_id(rec, i)\n        out.append(item)\n    return out\n\n\ndef compute_split(records: list[dict], test_size: int = 10,\n                  val_frac: float = 0.15, seed: int = 42) -> dict:\n    \"\"\"\n    Deterministically split annotated records into test / val / train.\n\n    test_size : number of held-out test records (default 10).\n    val_frac  : fraction of the *remaining* pool used for validation\n                (always at least 1 record so eval_loss can be computed).\n    Returns {\"train\": [...], \"val\": [...], \"test\": [...]} with records sorted by\n    original index within each split.\n    \"\"\"\n    annotated = _annotate(records)\n    n = len(annotated)\n    if test_size >= n:\n        raise ValueError(f\"test_size={test_size} >= dataset size {n}\")\n\n    order = list(range(n))\n    random.Random(seed).shuffle(order)\n\n    test_idx = order[:test_size]\n    pool_idx = order[test_size:]\n    val_count = max(1, round(val_frac * len(pool_idx)))\n    val_count = min(val_count, len(pool_idx) - 1)  # keep at least 1 for train\n    val_idx = pool_idx[:val_count]\n    train_idx = pool_idx[val_count:]\n\n    by_idx = {item[\"idx\"]: item for item in annotated}\n    pick = lambda idxs: [by_idx[i] for i in sorted(idxs)]\n    return {\"train\": pick(train_idx), \"val\": pick(val_idx), \"test\": pick(test_idx)}\n\n\n# ── Persisted test-set artifact ────────────────────────────────────────────────\n\n# Fields we keep in test_inputs.jsonl. `output` is the reference completion (for\n# eval comparison only — never fed to the model as input).\nTEST_FIELDS = (\"idx\", \"source_id\", \"instruction\", \"input\", \"output\")\n\n\ndef write_test_inputs(path: str | Path, test_records: list[dict]) -> None:\n    \"\"\"Persist the held-out test records to a JSONL artifact.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        for rec in test_records:\n            row = {k: rec.get(k, \"\") for k in TEST_FIELDS}\n            f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef read_test_inputs(path: str | Path) -> list[dict]:\n    \"\"\"Read the held-out test artifact.\"\"\"\n    return load_records(path)\n\n\ndef ids_of(records: list[dict]) -> set:\n    \"\"\"Set of canonical ids (idx, source_id) tuples for intersection checks.\"\"\"\n    out = set()\n    for i, rec in enumerate(records):\n        out.add((rec.get(\"idx\", i), rec.get(\"source_id\") or source_id(rec, i)))\n    return out\n\n\ndef ensure_test_inputs(data_path: str | Path,\n                       test_inputs_path: str | Path = DEFAULT_TEST_INPUTS,\n                       test_size: int = 10, val_frac: float = 0.15,\n                       seed: int = 42) -> dict:\n    \"\"\"\n    Return the split for ``data_path``, creating ``test_inputs_path`` from the\n    deterministic split if it does not exist yet. When it already exists, the\n    test split is taken from the persisted artifact (source of truth) and the\n    train/val pool is the complement, so all callers agree.\n    \"\"\"\n    records = load_records(data_path)\n    split = compute_split(records, test_size=test_size, val_frac=val_frac, seed=seed)\n\n    test_path = Path(test_inputs_path)\n    if not test_path.exists():\n        write_test_inputs(test_path, split[\"test\"])\n        return split\n\n    # Artifact exists. If it matches the seed-derived split exactly, use the\n    # canonical split so train/val are identical regardless of who created the\n    # file first (finetuning vs make_test_inputs).\n    persisted_ids = {row.get(\"source_id\") for row in read_test_inputs(test_path)}\n    if persisted_ids == {r[\"source_id\"] for r in split[\"test\"]}:\n        return split\n\n    # A custom / hand-edited artifact: respect it and derive the complement.\n    print(f\"[i] {test_path} differs from the seed-derived split; using the persisted test set.\")\n    annotated = _annotate(records)\n    pool = [r for r in annotated if r[\"source_id\"] not in persisted_ids]\n    test = [r for r in annotated if r[\"source_id\"] in persisted_ids]\n    order = list(range(len(pool)))\n    random.Random(seed).shuffle(order)\n    val_count = max(1, min(round(val_frac * len(pool)), len(pool) - 1))\n    val_idx = set(order[:val_count])\n    val = [pool[i] for i in sorted(val_idx)]\n    train = [pool[i] for i in range(len(pool)) if i not in val_idx]\n    return {\"train\": train, \"val\": val, \"test\": test}\n", encoding='utf-8')
pathlib.Path("run_inference.py").write_text("\"\"\"\nrun_inference.py\n----------------\nLoad the QLoRA fine-tuned Mistral-7B adapter and generate tri-modal cyber attack\nscenarios for the held-out test inputs.\n\nThis is the importable, testable home for inference logic — the notebook\n(finetune_output_test.ipynb) just calls into here, so the inference prompt can\nnever drift from the training prompt: both use prompt_format.build_chat.\n\nKey correctness properties (vs. the old notebook):\n  * Uses the SHARED canonical prompt with the FULL instruction (not a one-liner).\n  * Sets eos_token_id + pad_token_id on every generate() call so the model stops.\n  * Decodes ONLY the newly generated tokens (out[0][n:]) — the saved text is the\n    completion, never the echoed prompt.\n\nUsage (run from scenario_generation/):\n    python fine_tuning/run_inference.py \\\n        --adapter fine_tuning/output \\\n        --test_inputs evaluation/test_inputs.jsonl \\\n        --out fine_tuning/output/inference_results.txt\n    python fine_tuning/run_inference.py --smoke      # CUDA-guarded no-op probe\n\"\"\"\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport prompt_format\nimport data_split\n\n\nBASE_MODEL = \"mistralai/Mistral-7B-Instruct-v0.3\"\nREQUIRED_HEADERS = [\"[NATURAL LANGUAGE]\", \"[MATHEMATICAL EXPRESSION]\", \"[ATTACK GRAPH]\"]\n\n\n# ── Model loading ─────────────────────────────────────────────────────────────\n\ndef load_models(base, adapter, trust_remote_code=False, with_adapter=True):\n    \"\"\"Load the 4-bit base model and (optionally) attach the LoRA adapter.\"\"\"\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n    from peft import PeftModel\n\n    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    compute_dtype = torch.bfloat16 if bf16 else torch.float16\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=compute_dtype,\n    )\n\n    print(f\"[+] Loading base model ({'bf16' if bf16 else 'fp16'} compute): {base}\")\n    model = AutoModelForCausalLM.from_pretrained(\n        base, quantization_config=bnb_config, device_map=\"auto\",\n        trust_remote_code=trust_remote_code,\n    )\n    tokenizer = AutoTokenizer.from_pretrained(base, trust_remote_code=trust_remote_code)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    if with_adapter:\n        print(f\"[+] Attaching adapter: {adapter}\")\n        model = PeftModel.from_pretrained(model, adapter)\n\n    model.config.use_cache = True\n    model.eval()\n    return model, tokenizer\n\n\n# ── Generation ────────────────────────────────────────────────────────────────\n\ndef generation_kwargs(tokenizer, max_new_tokens, do_sample, temperature, top_p, repetition_penalty):\n    \"\"\"Assemble generate() kwargs; sampling params only included when sampling.\"\"\"\n    kw = dict(\n        max_new_tokens=max_new_tokens,\n        eos_token_id=tokenizer.eos_token_id,   # fix B2: stop on EOS\n        pad_token_id=tokenizer.pad_token_id,   # fix B2\n        do_sample=do_sample,\n        repetition_penalty=repetition_penalty,\n    )\n    if do_sample:\n        kw[\"temperature\"] = temperature\n        kw[\"top_p\"] = top_p\n    return kw\n\n\ndef generate_one(model, tokenizer, record, gen_kwargs):\n    \"\"\"Generate the completion for one record; returns (text, new_token_ids).\"\"\"\n    import torch\n\n    # build_chat already embeds BOS, so tokenize with add_special_tokens=False.\n    prompt = prompt_format.build_chat(record, tokenizer, for_training=False)\n    inputs = tokenizer(prompt, return_tensors=\"pt\", add_special_tokens=False).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n    with torch.no_grad():\n        out = model.generate(**inputs, **gen_kwargs)\n    new_ids = out[0][n:]                                  # fix I2: only new tokens\n    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()\n    return text, new_ids.tolist()\n\n\ndef headers_in_order(text):\n    pos = [text.find(h) for h in REQUIRED_HEADERS]\n    return all(p >= 0 for p in pos) and pos == sorted(pos), pos\n\n\n# ── Test-input loading ────────────────────────────────────────────────────────\n\ndef load_records(args):\n    \"\"\"Held-out test inputs by default; a single dataset index if --index given.\"\"\"\n    if args.index is not None:\n        recs = data_split.load_records(args.data)\n        return [dict(recs[args.index], source_id=data_split.source_id(recs[args.index], args.index))]\n    return data_split.read_test_inputs(args.test_inputs)\n\n\n# ── Main run ──────────────────────────────────────────────────────────────────\n\ndef run(args) -> int:\n    records = load_records(args)\n    if args.samples:\n        records = records[: args.samples]\n    print(f\"[+] {len(records)} record(s) to generate\")\n\n    model, tokenizer = load_models(args.base, args.adapter, args.trust_remote_code, with_adapter=True)\n    gk = generation_kwargs(tokenizer, args.max_new_tokens, args.do_sample,\n                           args.temperature, args.top_p, args.repetition_penalty)\n\n    base_model = None\n    if args.compare:\n        base_model, _ = load_models(args.base, args.adapter, args.trust_remote_code, with_adapter=False)\n\n    out_path = Path(args.out)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    with out_path.open(\"w\", encoding=\"utf-8\") as f:\n        f.write(\"TRI-MODAL CYBER ATTACK SCENARIO GENERATION\\n\")\n        f.write(\"=\" * 80 + \"\\n\\n\")\n        for i, rec in enumerate(records, 1):\n            sid = rec.get(\"source_id\", f\"#{i}\")\n            print(f\"[+] ({i}/{len(records)}) {sid}\")\n            text, _ = generate_one(model, tokenizer, rec, gk)\n\n            f.write(\"=\" * 80 + \"\\n\")\n            f.write(f\"SAMPLE {i}  (source_id: {sid})\\n\")\n            f.write(\"-\" * 80 + \"\\n\\n\")\n            f.write(\"INPUT:\\n\" + rec.get(\"input\", \"\") + \"\\n\\n\")\n            if args.compare and base_model is not None:\n                base_text, _ = generate_one(base_model, tokenizer, rec, gk)\n                f.write(\"BASELINE OUTPUT:\\n\" + base_text + \"\\n\\n\")\n                f.write(\"FINE-TUNED OUTPUT:\\n\" + text + \"\\n\\n\")\n            else:\n                f.write(\"OUTPUT:\\n\" + text + \"\\n\\n\")\n\n    print(f\"[✓] Saved to: {out_path}\")\n    return 0\n\n\ndef run_smoke(args) -> int:\n    \"\"\"CUDA-guarded one-sample self-test (fix I acceptance).\"\"\"\n    try:\n        import torch\n    except ImportError:\n        print(\"[smoke] PyTorch not installed — inference smoke is a NO-OP here.\")\n        print(\"        On Colab with a GPU it generates one held-out sample and\")\n        print(\"        checks all three headers appear in order and EOS is hit.\")\n        return 0\n    if not torch.cuda.is_available():\n        print(\"[smoke] No CUDA GPU — inference smoke is a NO-OP.\")\n        return 0\n\n    records = data_split.read_test_inputs(args.test_inputs)\n    model, tokenizer = load_models(args.base, args.adapter, args.trust_remote_code, with_adapter=True)\n    gk = generation_kwargs(tokenizer, args.max_new_tokens, do_sample=False,\n                           temperature=args.temperature, top_p=args.top_p,\n                           repetition_penalty=args.repetition_penalty)\n    text, new_ids = generate_one(model, tokenizer, records[0], gk)\n    ok_headers, pos = headers_in_order(text)\n    eos_hit = tokenizer.eos_token_id in new_ids\n    print(f\"[smoke] headers in order: {ok_headers} {pos}\")\n    print(f\"[smoke] EOS before cap  : {eos_hit} ({len(new_ids)}/{args.max_new_tokens} tokens)\")\n    ok = ok_headers and eos_hit\n    print(f\"[smoke] RESULT: {'PASS' if ok else 'FAIL'}\")\n    return 0 if ok else 1\n\n\ndef build_arg_parser():\n    p = argparse.ArgumentParser(description=\"Run inference with the fine-tuned adapter\")\n    p.add_argument(\"--base\", default=BASE_MODEL)\n    p.add_argument(\"--adapter\", default=\"fine_tuning/output\",\n                   help=\"Path to the saved LoRA adapter folder\")\n    p.add_argument(\"--test_inputs\", default=data_split.DEFAULT_TEST_INPUTS,\n                   help=\"Held-out test inputs JSONL\")\n    p.add_argument(\"--data\", default=\"dataset/training_data.jsonl\",\n                   help=\"Dataset (only used with --index)\")\n    p.add_argument(\"--index\", type=int, default=None,\n                   help=\"Generate for a specific dataset index instead of the test set\")\n    p.add_argument(\"--samples\", type=int, default=0, help=\"Limit number of samples (0 = all)\")\n    p.add_argument(\"--compare\", action=\"store_true\", help=\"Also generate base (no adapter) output\")\n    p.add_argument(\"--out\", default=\"fine_tuning/output/inference_results.txt\")\n    p.add_argument(\"--trust_remote_code\", action=\"store_true\")\n\n    # Generation controls (fix I3 defaults).\n    p.add_argument(\"--max_new_tokens\", type=int, default=1280, help=\"1024–1536 recommended\")\n    p.add_argument(\"--do_sample\", action=\"store_true\", default=True)\n    p.add_argument(\"--greedy\", dest=\"do_sample\", action=\"store_false\",\n                   help=\"Disable sampling (deterministic)\")\n    p.add_argument(\"--temperature\", type=float, default=0.7)\n    p.add_argument(\"--top_p\", type=float, default=0.9)\n    p.add_argument(\"--repetition_penalty\", type=float, default=1.1)\n    p.add_argument(\"--smoke\", action=\"store_true\", help=\"CUDA-guarded one-sample self-test\")\n    return p\n\n\ndef main():\n    args = build_arg_parser().parse_args()\n    if args.smoke:\n        sys.exit(run_smoke(args))\n    sys.exit(run(args))\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("test_inputs.jsonl").write_text("{\"idx\": 8, \"source_id\": \"C0044\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: Juicy Mix (C0044)\\nINDUSTRY: Electricity, gas, steam and AC supply (ISIC D)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1555.004 - Windows Credential Manager, T1059.005 - Visual Basic, T1555.003 - Credentials from Web Browsers, T1518 - Software Discovery, T1082 - System Information Discovery\\nSURFACES: Application runtime environment, Credential stores / memory / network traffic, Internal network / cloud metadata services\", \"output\": \"[NATURAL LANGUAGE]\\nThe attack begins with a spear-phishing email targeting employees of the electricity provider, Juicy Mix. The email contains a malicious attachment disguised as a legitimate document related to grid maintenance. Upon opening, the attachment executes a Visual Basic script (T1059.005) that performs initial reconnaissance, gathering system information (T1082) such as operating system version, installed software, and network configuration. This reconnaissance identifies a vulnerable web application server running an outdated content management system within the cloud environment. The script then attempts to harvest credentials stored in web browsers (T1555.003) and the Windows Credential Manager (T1555.004) on the compromised host.\\n\\nSuccessful credential harvesting provides the attacker with credentials for a low-privilege user account with access to the cloud environment. Leveraging these credentials, the attacker gains initial access to the cloud console. Software discovery (T1518) within the cloud environment reveals the presence of an improperly configured Identity and Access Management (IAM) role assigned to the web application server. This IAM role possesses overly permissive permissions, allowing the attacker to assume the role and escalate privileges. The attacker assumes the IAM role, granting them broader access to cloud resources, including storage buckets containing sensitive operational data and system logs.\\n\\nThe attacker then utilizes the elevated privileges to access and exfiltrate sensitive data from the storage buckets, including grid schematics, customer information, and operational procedures. The exfiltration is performed incrementally over a period of time to avoid detection, utilizing a combination of legitimate cloud APIs and encrypted channels. Simultaneously, the attacker deploys a lightweight backdoor, similar to Mango, onto the web application server to maintain persistent access to the environment. This backdoor allows for remote command execution and further reconnaissance.\\n\\nThe attack concludes with the successful exfiltration of critical data and the establishment of a persistent foothold within the cloud infrastructure. The compromised data could be used for sabotage, extortion, or espionage. The persistent backdoor allows the attacker to monitor the environment, gather further intelligence, and potentially launch future attacks. The initial misconfiguration of the IAM role and the lack of robust credential management practices were key factors in the success of this attack.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Employee Workstation, n2: Web Application Server, n3: Cloud Console, n4: Storage Bucket, n5: IAM Role}\\nS = {s1: Weak Credential Hygiene, s2: Outdated Software, s3: Overly Permissive IAM Role, s4: Lack of Multi-Factor Authentication}\\nV = {v1: T1059.005 - Execute VB Script, v2: T1082 - System Information Discovery, v3: T1555.004 - Windows Credential Manager, v4: T1555.003 - Credentials from Web Browsers, v5: T1518 - Software Discovery, v6: Privilege Escalation - Assume IAM Role}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n3 -> n5, p5: n5 -> n4}\\nFS = {Data Exfiltration, Persistent Access}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Employee Workstation [T1059.005] -> Web Application Server\\nStage 2 (Reconnaissance): Web Application Server [T1082] -> System Information Discovery\\nStage 3 (Credential Access): Web Application Server [T1555.004] -> Windows Credential Manager\\nStage 4 (Credential Access): Web Application Server [T1555.003] -> Credentials from Web Browsers\\nStage 5 (Initial Access - Cloud): Web Application Server -> Cloud Console\\nStage 6 (Discovery): Cloud Console [T1518] -> Software Discovery\\nStage 7 (Privilege Escalation): Cloud Console -> IAM Role\\nStage 8 (Lateral Movement): IAM Role -> Storage Bucket\\nStage 9 (Exfiltration): Storage Bucket -> Attacker Controlled Infrastructure\\nStage 10 (Persistence): Web Application Server -> Backdoor Deployment\\nFinal Impact: Sensitive data exfiltrated, persistent backdoor established for continued access and potential sabotage.\"}\n{\"idx\": 12, \"source_id\": \"C0047\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: RedDelta Modified PlugX Infection Chain Operations (C0047)\\nINDUSTRY: Information technology service activities (ISIC J62)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1588.004 - Digital Certificates, T1027.013 - Encrypted/Encoded File, T1608.001 - Upload Malware, T1218.014 - MMC, T1082 - System Information Discovery\\nSURFACES: Application runtime environment, Internal network / cloud metadata services, Security tooling / logging infrastructure\", \"output\": \"[NATURAL LANGUAGE]\\nThe attack begins with a highly targeted phishing campaign (T1566.001 - Phishing) against employees within the IT service provider, leveraging a digitally signed document (T1588.004 - Digital Certificates) to bypass initial email security checks. This document contains an embedded, encrypted executable (T1027.013 - Encrypted/Encoded File) disguised as a legitimate software update. Upon execution, the payload, a variant of RedDelta Modified PlugX, establishes a foothold on the victim’s workstation. The malware utilizes MMC (T1218.014 - MMC) to execute further malicious code and perform system information discovery (T1082 - System Information Discovery) to map the internal network and identify potential targets, focusing on systems with access to cloud infrastructure.\\n\\nLateral movement is achieved by exploiting weak credentials and leveraging the compromised workstation as a pivot point. The attacker utilizes stolen credentials to access cloud metadata services, specifically targeting IAM roles with overly permissive policies. This allows them to enumerate cloud resources (T1082) and identify vulnerable instances, such as EC2 instances running outdated software or with publicly accessible storage buckets. The attacker then uploads additional malware (T1608.001 - Upload Malware) to these instances, potentially using compromised developer accounts or exploiting misconfigured CI/CD pipelines.\\n\\nOnce inside the cloud environment, the attacker focuses on identifying and accessing sensitive data stored in databases and object storage. They leverage their elevated privileges to exfiltrate data (T1041 - Exfiltration Over C2 Channel) to an external command and control server. The attacker specifically targets customer data, intellectual property, and configuration details related to the IT service provider’s clients. The compromised cloud infrastructure is also used to launch further attacks against the provider’s customers, leveraging the provider’s trusted network as a launching pad.\\n\\nThe attack is exacerbated by inadequate security logging and monitoring. The attacker actively attempts to clear logs (T1070.001 - Indicator Removal on Host) to cover their tracks and delay detection. The lack of robust intrusion detection systems and security information and event management (SIEM) solutions allows the attacker to maintain persistence and operate undetected for an extended period. The IT service provider’s reliance on default cloud configurations and a lack of regular security audits contribute to the success of the attack.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Employee Workstation, n2: Email Server, n3: Cloud Metadata Service, n4: EC2 Instance, n5: Object Storage, n6: Database, n7: C2 Server}\\nS = {s1: Weak Credentials, s2: Overly Permissive IAM Roles, s3: Outdated Software, s4: Misconfigured CI/CD Pipeline, s5: Inadequate Logging}\\nV = {v1: T1566.001 - Phishing, v2: T1588.004 - Digital Certificates, v3: T1027.013 - Encrypted/Encoded File, v4: T1218.014 - MMC, v5: T1082 - System Information Discovery, v6: T1608.001 - Upload Malware, v7: T1041 - Exfiltration Over C2 Channel, v8: T1070.001 - Indicator Removal on Host}\\nP = {p1: n2 -> n1 -> n3, p2: n1 -> n4, p3: n3 -> n4 -> n5, p4: n3 -> n4 -> n6, p5: n4 -> n7}\\nFS = {Data Exfiltration, Compromised Customer Data, Lateral Movement to Client Networks}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Email Server [T1566.001] -> Employee Workstation\\nStage 2 (Execution): Employee Workstation [T1588.004, T1027.013, T1218.014] -> Malware Installation\\nStage 3 (Discovery): Employee Workstation [T1082] -> Internal Network Mapping\\nStage 4 (Lateral Movement): Employee Workstation -> Cloud Metadata Service (via stolen credentials [S1])\\nStage 5 (Privilege Escalation): Cloud Metadata Service [S2] -> EC2 Instance\\nStage 6 (Deployment): EC2 Instance [T1608.001] -> Object Storage & Database\\nStage 7 (Exfiltration): EC2 Instance -> C2 Server [T1041]\\nStage 8 (Persistence & Obfuscation): EC2 Instance [T1070.001] -> Log Manipulation\\nFinal Impact: Sensitive data exfiltrated, customer data compromised, potential for further attacks against clients.\"}\n{\"idx\": 15, \"source_id\": \"C0016\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: Operation Dust Storm (C0016)\\nINDUSTRY: Financial and insurance activities (ISIC K)\\nATTACK LEVEL: HIGH\\nTECHNIQUES: T1059.005 - Visual Basic, T1204.002 - Malicious File, T1027.013 - Encrypted/Encoded File, T1518 - Software Discovery, T1189 - Drive-by Compromise\\nSURFACES: Application runtime environment, Internal network / cloud metadata services, Internet-facing services / user endpoints, Security tooling / logging infrastructure\", \"output\": \"[NATURAL LANGUAGE]\\nOperation Dust Storm targets a financial institution, initiating the attack with a drive-by compromise (T1189) via a compromised advertising network displaying malicious advertisements on websites frequented by employees. These ads redirect users to a landing page hosting an obfuscated Visual Basic script (T1059.005) disguised as a legitimate financial report. Upon execution, the script downloads and executes an encrypted executable (T1027.013) – a custom malware loader – onto the victim’s workstation. This loader performs software discovery (T1518) to identify installed security solutions and cloud agents, attempting to evade detection.\\n\\nOnce established, the malware leverages stolen credentials (obtained through keylogging or credential dumping – not explicitly a listed technique but implied) to gain initial access to the cloud environment. The attacker targets misconfigured IAM roles with excessive permissions, specifically a role allowing read/write access to sensitive data stored in an Amazon S3 bucket containing customer financial records. Lateral movement occurs by exploiting trust relationships between cloud services and utilizing the compromised IAM role to access other resources, including virtual machines hosting internal applications. The attacker then uses the compromised VMs as a staging ground for further reconnaissance and data collection.\\n\\nThe attacker focuses on identifying and accessing databases containing Personally Identifiable Information (PII) and financial data. They utilize SQL injection techniques (not explicitly listed but a common tactic) to extract data directly from the databases. Simultaneously, they attempt to disable or tamper with security logging and monitoring systems to hinder detection and incident response. This involves targeting cloud logging services like AWS CloudTrail and CloudWatch Logs, attempting to delete or modify log entries.\\n\\nData exfiltration is achieved by compressing the stolen data and uploading it to a cloud storage service controlled by the attacker, utilizing the compromised IAM role’s permissions. The attacker employs steganography within seemingly innocuous image files to further conceal the exfiltrated data, making it harder to detect during network traffic analysis. The long-term goal is not immediate financial gain but rather sustained espionage, gathering intelligence on the financial institution’s operations, customers, and security posture.\\n\\nThe attack concludes with the attacker establishing persistence within the cloud environment by creating backdoors and maintaining access through compromised accounts and systems. This allows them to continue monitoring the organization and potentially launch future attacks. The financial institution suffers significant reputational damage, financial losses due to regulatory fines, and potential legal liabilities related to the data breach.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: User Workstation, n2: Advertising Network, n3: Cloud IAM Role, n4: S3 Bucket, n5: Virtual Machine, n6: Database Server, n7: Cloud Logging Service, n8: Attacker Controlled Storage}\\nS = {s1: Misconfigured IAM Role, s2: Weak Password Policy, s3: Unpatched Software, s4: Lack of Multi-Factor Authentication, s5: Insufficient Logging}\\nV = {v1: T1189 - Drive-by Compromise, v2: T1059.005 - Visual Basic, v3: T1027.013 - Encrypted/Encoded File, v4: T1518 - Software Discovery, v5: Credential Access, v6: Lateral Movement, v7: Data Exfiltration}\\nP = {p1: n2 -> n1 -> n3 -> n4, p2: n3 -> n5 -> n6, p3: n3 -> n7, p4: n4 -> n8, p5: n6 -> n8}\\nFS = {Data Exfiltration, Persistence Established, Intelligence Gathering}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Advertising Network [T1189] -> User Workstation\\nStage 2 (Execution): User Workstation [T1059.005] -> User Workstation\\nStage 3 (Defense Evasion): User Workstation [T1027.013] -> User Workstation\\nStage 4 (Discovery): User Workstation [T1518] -> Cloud IAM Role\\nStage 5 (Credential Access): User Workstation -> Cloud IAM Role (Implied - Credential Dumping)\\nStage 6 (Lateral Movement): Cloud IAM Role -> S3 Bucket\\nStage 7 (Lateral Movement): Cloud IAM Role -> Virtual Machine\\nStage 8 (Collection): Virtual Machine -> Database Server\\nStage 9 (Exfiltration): Database Server -> Cloud IAM Role -> Attacker Controlled Storage\\nStage 10 (Defense Evasion): Cloud IAM Role -> Cloud Logging Service (Tampering)\\nFinal Impact: Sensitive customer financial data exfiltrated, persistent access established for ongoing espionage, and significant reputational and financial damage to the financial institution.\"}\n{\"idx\": 30, \"source_id\": \"C0045\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: ShadowRay (C0045)\\nINDUSTRY: Human health and social work (ISIC Q)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1027.013 - Encrypted/Encoded File, T1016 - System Network Configuration Discovery, T1496.001 - Compute Hijacking, T1190 - Exploit Public-Facing Application, T1068 - Exploitation for Privilege Escalation\\nSURFACES: Internal network / cloud metadata services, Internet-facing services / user endpoints, OS / IAM permission model, Production systems / backups / data integrity\", \"output\": \"[NATURAL LANGUAGE]\\nThe attack begins with ShadowRay operatives targeting a publicly exposed web application within the human health organization’s infrastructure (T1190). This application, used for patient appointment scheduling, contains a known vulnerability – a deserialization flaw – allowing the attackers to inject a malicious payload disguised as a legitimate user input. This payload is delivered via an encrypted file (T1027.013) to evade initial detection by signature-based security tools. Successful exploitation grants initial access to a low-privilege web server instance running within the cloud provider’s environment.\\n\\nOnce inside, the attackers leverage system network configuration discovery (T1016) to map the internal network, identifying other systems and services accessible from the compromised web server. This reconnaissance reveals the presence of cloud metadata services accessible via the instance’s role, which are not adequately restricted. Exploiting this misconfiguration allows the attackers to obtain temporary credentials with elevated privileges. These credentials are then used to launch compute hijacking attacks (T1496.001) against other cloud instances, specifically targeting those running database servers containing sensitive patient data.\\n\\nLateral movement continues as the attackers identify a vulnerable system running an outdated operating system. They exploit this vulnerability (T1068) to gain root access on that system. From this highly privileged position, they are able to access and exfiltrate the patient database, archiving the data into encrypted files before transferring it to an external command-and-control server. The attackers also deploy a persistent backdoor for future access, ensuring continued control over the compromised environment.\\n\\nThe organization’s backups are also targeted. The attackers identify a misconfigured backup storage bucket with overly permissive access controls. They compromise the backups, encrypting them with a strong encryption algorithm, and demand a ransom for their decryption. This dual-pronged attack – data exfiltration and backup encryption – maximizes the impact and pressure on the organization. The attackers specifically target data covered by HIPAA regulations, increasing the severity of the breach and potential legal ramifications.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Public Web Application, n2: Web Server Instance, n3: Cloud Metadata Service, n4: Database Server, n5: Vulnerable System, n6: Backup Storage Bucket, n7: External C2 Server}\\nS = {s1: Deserialization Vulnerability, s2: Unrestricted Metadata Access, s3: Outdated OS, s4: Permissive Backup Bucket Access}\\nV = {v1: T1190 - Exploit Public-Facing Application, v2: T1027.013 - Encrypted/Encoded File, v3: T1016 - System Network Configuration Discovery, v4: T1496.001 - Compute Hijacking, v5: T1068 - Exploitation for Privilege Escalation}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n2 -> n5, p5: n5 -> n6, p6: n4 -> n7, p7: n6 -> n7}\\nFS = {Data Exfiltration, Backup Encryption, Persistent Access}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Public Web Application [T1190] -> Web Server Instance\\nStage 2 (Discovery): Web Server Instance [T1016] -> Cloud Metadata Service\\nStage 3 (Privilege Escalation): Cloud Metadata Service -> Database Server [T1496.001]\\nStage 4 (Lateral Movement): Web Server Instance -> Vulnerable System [T1068]\\nStage 5 (Privilege Escalation): Vulnerable System [T1068] -> Backup Storage Bucket\\nStage 6 (Data Exfiltration): Database Server -> External C2 Server [T1027.013]\\nStage 7 (Data Exfiltration): Backup Storage Bucket -> External C2 Server [T1027.013]\\nFinal Impact: Sensitive patient data exfiltrated, backups encrypted, persistent backdoor established, resulting in significant financial and reputational damage to the healthcare organization.\"}\n{\"idx\": 32, \"source_id\": \"C0032\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: C0032 (C0032)\\nINDUSTRY: Information technology service activities (ISIC J62)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1583.003 - Virtual Private Server, T1003.001 - LSASS Memory, T1078 - Valid Accounts, T1571 - Non-Standard Port, T1588.002 - Tool\\nSURFACES: Credential stores / memory / network traffic, Internet-facing services / user endpoints, OS / IAM permission model, Outbound network / DNS / HTTP channels\", \"output\": \"[NATURAL LANGUAGE]\\nThe attack begins with the compromise of a Virtual Private Server (VPS) (T1583.003) used by the IT service provider for development and testing. This VPS, while not directly hosting production data, has network access to the internal network via a misconfigured VPN connection. The attackers leverage this foothold to scan the internal network for exposed services listening on non-standard ports (T1571), identifying a legacy application server running a vulnerable version of a web application. They exploit a known vulnerability in this application to gain initial access to a valid account (T1078) with limited privileges.\\n\\nOnce inside the network, the attackers focus on lateral movement. They identify a domain controller and attempt to dump credentials from its memory using LSASS Memory (T1003.001) via a tool (T1588.002) uploaded to the compromised application server. Successful credential harvesting provides them with domain administrator credentials. With these elevated privileges, they gain access to cloud management consoles, specifically targeting IAM roles and permissions. A critical misconfiguration allows a low-privilege IAM role to assume a higher-privilege role with access to sensitive data stored in a cloud object storage service (e.g., AWS S3, Azure Blob Storage).\\n\\nThe attackers then utilize the compromised IAM role to enumerate and download sensitive data from the object storage, including customer Personally Identifiable Information (PII) and proprietary source code. They compress and encrypt the exfiltrated data before sending it out through the initially compromised VPS using outbound network channels. The attackers attempt to cover their tracks by deleting logs on compromised systems, but incomplete log deletion leaves traces of their activity.\\n\\nThe IT service provider’s lack of network segmentation, coupled with the misconfigured VPN and overly permissive IAM roles, facilitated the attack. The use of a VPS as an initial access point allowed the attackers to bypass perimeter defenses and establish a persistent presence within the network. The successful LSASS memory dump and subsequent privilege escalation were key to gaining access to the cloud environment and exfiltrating sensitive data.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: VPS, n2: Application Server, n3: Domain Controller, n4: Cloud Management Console, n5: Object Storage}\\nS = {s1: Misconfigured VPN, s2: Vulnerable Application, s3: Overly Permissive IAM Role, s4: Incomplete Log Deletion}\\nV = {v1: T1583.003 - VPS Compromise, v2: T1571 - Port Scanning, v3: T1078 - Valid Account Exploitation, v4: T1003.001 - LSASS Memory Dump, v5: T1588.002 - Tool Transfer, v6: IAM Role Assumption}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n4 -> n5}\\nFS = {Data Exfiltration - PII and Source Code}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): VPS [T1583.003] -> Application Server [T1571]\\nStage 2 (Exploitation): Application Server [T1078] -> Domain Controller [T1588.002]\\nStage 3 (Credential Access): Domain Controller [T1003.001] -> Cloud Management Console\\nStage 4 (Privilege Escalation): Cloud Management Console [T1588.002] -> IAM Role Assumption\\nStage 5 (Lateral Movement): Cloud Management Console -> Object Storage\\nStage 6 (Exfiltration): Object Storage -> VPS [T1583.003]\\nFinal Impact: Sensitive data (PII and source code) exfiltrated from Object Storage via the compromised VPS.\"}\n{\"idx\": 49, \"source_id\": \"C0052\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: SPACEHOP Activity (C0052)\\nINDUSTRY: Information technology service activities (ISIC J62)\\nATTACK LEVEL: HIGH\\nTECHNIQUES: T1190 - Exploit Public-Facing Application, T1588.002 - Tool, T1090.003 - Multi-hop Proxy, T1583.003 - Virtual Private Server\\nSURFACES: Internet-facing services / user endpoints, Outbound network / DNS / HTTP channels\", \"output\": \"[NATURAL LANGUAGE]\\nThe attack begins with the exploitation of a publicly facing application within the IT service provider’s infrastructure (T1190). This application, a customer-facing web portal used for managing service requests, suffers from a SQL injection vulnerability. The attackers, leveraging techniques consistent with SPACEHOP activity, scan for and identify this vulnerability using automated tools (T1588.002 - Tool, specifically web vulnerability scanners). Successful exploitation allows initial code execution on the web server, granting a foothold within the network. To obfuscate their origin and evade detection, the attackers establish a multi-hop proxy chain (T1090.003) utilizing compromised servers and publicly available proxy services. This proxy chain routes traffic through multiple geographic locations, masking the true source of the attack.\\n\\nOnce inside, lateral movement focuses on identifying and compromising internal systems with access to sensitive customer data. The attackers utilize the initial foothold to scan the internal network for other vulnerable systems, specifically targeting systems running outdated software or with weak authentication protocols. They identify a domain controller with a known vulnerability and leverage this to obtain domain administrator credentials. This privilege escalation allows them to access and control a significant portion of the IT service provider’s infrastructure. The attackers then pivot to cloud resources, specifically targeting misconfigured AWS S3 buckets containing customer data backups and configuration files.\\n\\nThe attackers exploit a common misconfiguration: publicly accessible S3 buckets. These buckets, intended for internal use, lack proper access controls, allowing the attackers to enumerate and download sensitive data. This data includes personally identifiable information (PII), financial records, and intellectual property belonging to the IT service provider’s clients. The exfiltration is conducted slowly and deliberately, utilizing the multi-hop proxy chain to avoid triggering anomaly detection systems. Data is compressed and encrypted before being transferred to a VPS (T1583.003) controlled by the attackers, serving as a staging area for further analysis and potential use.\\n\\nThe impact of this attack is significant. The IT service provider suffers a major data breach, resulting in financial losses, reputational damage, and potential legal liabilities. Clients are notified of the breach, leading to loss of trust and potential business disruption. The attackers maintain persistence within the network through backdoors installed on compromised systems, allowing for potential future attacks. The compromised domain administrator credentials could be used to launch further attacks against the IT service provider’s clients, expanding the scope of the breach. The attackers’ use of commercially leased VPS infrastructure and multi-hop proxies makes attribution and remediation particularly challenging.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Public Facing Web Application, n2: Web Server, n3: Internal Network, n4: Domain Controller, n5: AWS S3 Bucket, n6: VPS}\\nS = {s1: SQL Injection Vulnerability, s2: Weak Authentication, s3: Publicly Accessible S3 Bucket, s4: Outdated Software}\\nV = {v1: T1190 - Exploit Public-Facing Application, v2: T1588.002 - Tool (Vulnerability Scanner), v3: T1090.003 - Multi-hop Proxy, v4: T1583.003 - Virtual Private Server, v5: T1078 - Valid Accounts}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n4 -> n5, p5: n5 -> n6}\\nFS = {Data Exfiltration - PII, Financial Records, Intellectual Property}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Public Facing Web Application [T1190] -> Web Server\\nStage 2 (Discovery): Web Server [T1588.002] -> Internal Network\\nStage 3 (Lateral Movement): Internal Network [T1078] -> Domain Controller\\nStage 4 (Privilege Escalation): Domain Controller [T1078] -> AWS S3 Bucket\\nStage 5 (Collection): AWS S3 Bucket -> VPS [T1583.003]\\nStage 6 (Exfiltration): VPS [T1090.003] -> Attacker Infrastructure\\nFinal Impact: Sensitive customer data (PII, financial records, intellectual property) exfiltrated and attacker maintains persistence.\"}\n{\"idx\": 52, \"source_id\": \"G1014\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: LuminousMoth (G1014)\\nINDUSTRY: Public administration and defence (ISIC O)\\nATTACK LEVEL: HIGH\\nTECHNIQUES: T1588.004 - Digital Certificates, T1588.002 - Tool, T1557.002 - ARP Cache Poisoning, T1608.001 - Upload Malware, T1112 - Modify Registry\\nSURFACES: Credential stores / memory / network traffic, Data egress points / cloud APIs, File system / cloud storage / databases, Security tooling / logging infrastructure\", \"output\": \"[NATURAL LANGUAGE]\\nLuminousMoth initiates the attack against a public administration entity by leveraging a compromised digital certificate (T1588.004) to gain initial access. This certificate, seemingly legitimate, is used to establish a trusted connection to a publicly facing web application hosted on a cloud provider (AWS, Azure, or GCP). The application suffers from a lack of robust input validation, allowing the attacker to upload a malicious tool (T1588.002) disguised as a legitimate software update. This tool is a custom backdoor written in Go, designed for stealthy persistence and lateral movement. Once executed, the backdoor establishes a command-and-control (C2) channel over HTTPS, blending in with normal network traffic.\\n\\nFollowing initial access, the attacker performs internal reconnaissance to map the network and identify high-value targets. Utilizing ARP Cache Poisoning (T1557.002) on the internal network, the attacker intercepts sensitive network traffic, including credentials used for accessing cloud resources. These credentials, often stored in plaintext or weakly encrypted configuration files, provide access to the organization’s cloud environment. The attacker then focuses on escalating privileges by exploiting a misconfigured IAM role with overly permissive access to sensitive data stores, specifically a cloud-based database containing citizen records. They further solidify their foothold by modifying the Windows Registry (T1112) on compromised systems to ensure persistence across reboots and hinder detection by security software.\\n\\nLateral movement occurs through exploiting trust relationships between cloud resources and leveraging the compromised credentials. The attacker identifies a server responsible for generating reports containing sensitive citizen data. They then utilize the compromised IAM role to access and download these reports directly from the cloud storage bucket (T1608.001). Simultaneously, the attacker targets the security information and event management (SIEM) system, attempting to disable logging or manipulate logs to cover their tracks.\\n\\nThe primary objective of LuminousMoth is data exfiltration. The attacker successfully exfiltrates a large volume of personally identifiable information (PII) from the cloud database and report files, compressing and encrypting the data before transferring it to a C2 server located outside the jurisdiction. The attack concludes with the attacker maintaining persistent access to the cloud environment, potentially for future espionage or disruption activities. The compromised systems are left as potential launchpads for further attacks against other government entities.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Public Web Application, n2: Internal Server, n3: Cloud Storage Bucket, n4: Cloud Database, n5: IAM Role, n6: SIEM System, n7: Attacker C2 Server}\\nS = {s1: Lack of Input Validation, s2: Weak Credential Storage, s3: Overly Permissive IAM Role, s4: Insufficient Logging, s5: ARP Spoofing Vulnerability}\\nV = {v1: T1588.004 - Certificate-Based Access, v2: T1588.002 - Tool Upload, v3: T1557.002 - ARP Cache Poisoning, v4: T1608.001 - Data Exfiltration, v5: T1112 - Registry Modification}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n2 -> n4, p4: n2 -> n5, p5: n5 -> n3, p6: n5 -> n4, p7: n2 -> n6}\\nFS = {Data Exfiltration of PII, Persistent Access to Cloud Environment}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Public Web Application [T1588.004] -> Internal Server\\nStage 2 (Execution): Internal Server [T1588.002] -> Establish C2 Channel\\nStage 3 (Reconnaissance): Internal Server [T1557.002] -> Intercept Network Traffic\\nStage 4 (Credential Access): Intercepted Traffic -> Obtain Cloud Credentials\\nStage 5 (Privilege Escalation): Cloud Credentials -> IAM Role [T1112] -> Modify Registry for Persistence\\nStage 6 (Lateral Movement): IAM Role -> Cloud Storage Bucket\\nStage 7 (Lateral Movement): IAM Role -> Cloud Database\\nStage 8 (Collection): Cloud Database [T1608.001] -> Exfiltrate PII to Attacker C2 Server\\nStage 9 (Command and Control): Attacker C2 Server -> Maintain Persistent Access\\nFinal Impact: Successful exfiltration of sensitive citizen data and establishment of long-term persistent access to the organization’s cloud infrastructure.\"}\n{\"idx\": 53, \"source_id\": \"G1051\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: Medusa Group (G1051)\\nINDUSTRY: Electricity, gas, steam and AC supply (ISIC D)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1585.001 - Social Media Accounts, T1518.001 - Security Software Discovery, T1559.001 - Component Object Model, T1018 - Remote System Discovery, T1190 - Exploit Public-Facing Application\\nSURFACES: Application runtime environment, Internal network / cloud metadata services, Internet-facing services / user endpoints, Production systems / backups / data integrity\", \"output\": \"[NATURAL LANGUAGE]\\nMedusa Group initiates the attack by leveraging social media accounts (T1585.001) of employees within the electricity supply company, identified through OSINT gathering. Spear phishing messages, disguised as industry news or safety alerts, are sent to targeted employees, containing malicious links. These links lead to a compromised, internet-facing web application (T1190) hosted on a cloud provider (AWS, Azure, or GCP). The application suffers from a known, but unpatched, vulnerability allowing for remote code execution. Successful exploitation grants initial access to a web server instance within the application runtime environment.\\n\\nOnce inside, the attackers perform security software discovery (T1518.001) to identify and bypass endpoint detection and response (EDR) solutions and firewalls. Utilizing the compromised web server, they conduct remote system discovery (T1018) to map the internal network and identify critical systems, including database servers storing customer data and operational technology (OT) systems controlling grid infrastructure. Lateral movement is achieved through exploiting the Component Object Model (T1559.001) on vulnerable Windows-based systems within the internal network, leveraging stolen credentials from the initial web server compromise.\\n\\nPrivilege escalation occurs by exploiting misconfigured IAM roles within the cloud environment. The attackers identify an IAM role with excessive permissions, allowing them to access and manipulate cloud metadata services. This access enables them to enumerate other cloud resources, including storage buckets containing backups of critical data. The attackers then exfiltrate sensitive data, including customer PII, operational data, and potentially intellectual property, to an external command-and-control server.\\n\\nTo maximize impact and pressure the organization, Medusa Group encrypts critical databases and backups, deploying ransomware across the compromised systems. The attackers leave a ransom note demanding payment in cryptocurrency for decryption keys. The attack specifically targets systems responsible for billing and customer service, disrupting operations and causing significant financial and reputational damage. The attackers also threaten to publicly release the exfiltrated data if the ransom is not paid, further amplifying the impact.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Employee Social Media Account, n2: Internet-Facing Web Application, n3: Web Server Instance, n4: Internal Network, n5: Windows System, n6: Database Server, n7: Cloud Metadata Service, n8: Storage Bucket, n9: OT System}\\nS = {s1: Unpatched Web Application Vulnerability, s2: Weak Social Engineering Awareness, s3: Misconfigured IAM Roles, s4: Exploitable Component Object Model, s5: Insufficient Network Segmentation}\\nV = {v1: T1585.001 - Spear Phishing Link, v2: T1190 - Exploit Public-Facing Application, v3: T1518.001 - Security Software Discovery, v4: T1018 - Remote System Discovery, v5: T1559.001 - Component Object Model Exploitation, v6: Privilege Escalation via IAM Role, v7: Data Exfiltration, v8: Ransomware Deployment}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n4 -> n5, p5: n5 -> n6, p6: n3 -> n7, p7: n7 -> n8, p8: n4 -> n9}\\nFS = {Data Exfiltration, System Encryption, Operational Disruption, Financial Loss}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Employee Social Media Account (T1585.001) -> Internet-Facing Web Application\\nStage 2 (Exploitation): Internet-Facing Web Application (T1190) -> Web Server Instance\\nStage 3 (Discovery): Web Server Instance (T1518.001) -> Internal Network\\nStage 4 (Lateral Movement): Internal Network (T1018) -> Windows System\\nStage 5 (Privilege Escalation): Windows System (T1559.001) -> Cloud Metadata Service\\nStage 6 (Data Access): Cloud Metadata Service -> Storage Bucket\\nStage 7 (Data Exfiltration): Storage Bucket -> External C2 Server (T1041)\\nStage 8 (Impact): Database Server -> Ransomware Deployment (T1486)\\nFinal Impact: Critical systems encrypted with ransomware, sensitive data exfiltrated, operational disruption, and significant financial loss.\"}\n{\"idx\": 57, \"source_id\": \"G1048\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: UNC3886 (G1048)\\nINDUSTRY: Public administration and defence (ISIC O)\\nATTACK LEVEL: HIGH\\nTECHNIQUES: T1588.004 - Digital Certificates, T1037.004 - RC Scripts, T1190 - Exploit Public-Facing Application, T1040 - Network Sniffing, T1562.003 - Impair Command History Logging\\nSURFACES: Credential stores / memory / network traffic, Internal network / cloud metadata services, Internet-facing services / user endpoints, OS / IAM permission model\", \"output\": \"[NATURAL LANGUAGE]\\nUNC3886 initiates the attack against a public administration entity by exploiting a publicly facing web application (T1190) used for citizen services. This application, hosted on a cloud provider’s compute service (e.g., AWS EC2, Azure Virtual Machines), suffers from a known, but unpatched, vulnerability allowing for remote code execution. The attackers deploy a malicious RC script (T1037.004) via this vulnerability, establishing an initial foothold within the network. This script is designed to perform reconnaissance, identifying internal network segments, cloud metadata service accessibility, and potential credential stores.\\n\\nOnce inside, UNC3886 leverages network sniffing (T1040) on the compromised server to capture credentials in transit, specifically targeting administrative accounts with access to the cloud environment’s IAM (Identity and Access Management) system. The attackers then utilize these stolen credentials to escalate privileges, gaining access to sensitive cloud resources. A key tactic involves exploiting overly permissive IAM roles assigned to service accounts, allowing them to assume higher-level permissions than necessary. Simultaneously, they impair command history logging (T1562.003) on compromised systems to hinder forensic analysis.\\n\\nLateral movement occurs through the cloud environment, targeting storage services (e.g., AWS S3, Azure Blob Storage) and databases (e.g., AWS RDS, Azure SQL Database) containing sensitive citizen data and government records. UNC3886 utilizes compromised credentials and IAM roles to access these resources. To maintain persistence and evade detection, they generate and deploy a rogue digital certificate (T1588.004) to sign malicious payloads and establish secure communication channels within the cloud infrastructure.\\n\\nThe attackers focus on exfiltrating data related to national security and critical infrastructure. They compress and encrypt the stolen data before transferring it out of the cloud environment using a covert channel, potentially leveraging legitimate cloud services like content delivery networks (CDNs) or email services to disguise the exfiltration traffic. The exfiltration is carefully timed to avoid triggering anomaly detection systems. The attackers also attempt to establish a long-term presence within the cloud environment by creating backdoors and modifying system configurations.\\n\\nThe ultimate goal is to gather intelligence on government policies, critical infrastructure vulnerabilities, and citizen data for espionage purposes. The attack demonstrates a sophisticated understanding of cloud infrastructure, IAM permissions, and network protocols, highlighting the need for robust security measures and continuous monitoring in public administration organizations.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Public Facing Web App, n2: Compromised Compute Instance, n3: Internal Network, n4: Cloud Metadata Service, n5: IAM System, n6: Storage Service, n7: Database, n8: CDN}\\nS = {s1: Unpatched Web App Vulnerability, s2: Overly Permissive IAM Roles, s3: Credentials in Transit, s4: Insufficient Logging}\\nV = {v1: T1190 - Exploit Public-Facing Application, v2: T1037.004 - RC Scripts, v3: T1040 - Network Sniffing, v4: T1562.003 - Impair Command History Logging, v5: T1588.004 - Digital Certificates}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n2 -> n4, p4: n4 -> n5, p5: n5 -> n6, p6: n5 -> n7, p7: n6 -> n8}\\nFS = {Data Exfiltration - Sensitive Citizen and Government Data}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Public Facing Web App [T1190] -> Compromised Compute Instance\\nStage 2 (Execution): Compromised Compute Instance [T1037.004] -> Internal Network\\nStage 3 (Reconnaissance): Internal Network [T1040] -> Cloud Metadata Service\\nStage 4 (Credential Access): Cloud Metadata Service [T1040] -> IAM System\\nStage 5 (Privilege Escalation): IAM System [T1562.003] -> Storage Service & Database\\nStage 6 (Lateral Movement): Storage Service & Database -> CDN\\nStage 7 (Command and Control): Compromised Compute Instance [T1588.004] -> CDN\\nStage 8 (Exfiltration): CDN -> External Server\\nFinal Impact: Successful exfiltration of sensitive citizen and government data, long-term persistence established within the cloud environment.\"}\n{\"idx\": 63, \"source_id\": \"G0117\", \"instruction\": \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\nYour task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\nCRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n3. The three section headers must appear EXACTLY as plain text on their own line:\\n   [NATURAL LANGUAGE]\\n   [MATHEMATICAL EXPRESSION]\\n   [ATTACK GRAPH]\\n4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n7. The attack graph must list numbered stages with plain text arrows (->).\", \"input\": \"SOURCE: Fox Kitten (G0117)\\nINDUSTRY: Electricity, gas, steam and AC supply (ISIC D)\\nATTACK LEVEL: MEDIUM\\nTECHNIQUES: T1136.001 - Local Account, T1027.013 - Encrypted/Encoded File, T1585.001 - Social Media Accounts, T1018 - Remote System Discovery, T1087.002 - Domain Account\\nSURFACES: Internal network / cloud metadata services, Internet-facing services / user endpoints, Security tooling / logging infrastructure, System configuration / registry / cron\", \"output\": \"[NATURAL LANGUAGE]\\nFox Kitten initiates the attack against an electricity provider by leveraging social media accounts (T1585.001) of disgruntled former employees. These accounts are used to spear-phish current employees with access to the organization’s cloud infrastructure, specifically targeting those with administrative privileges. The phishing emails contain an encrypted/encoded file (T1027.013) disguised as a legitimate document related to industry regulations. Once opened, the file executes a PowerShell script that establishes a foothold on the victim’s workstation. This initial access is then used for reconnaissance (T1018) to map the internal network and identify cloud resources, focusing on misconfigured IAM roles and exposed metadata services.\\n\\nLateral movement begins by exploiting a local account (T1136.001) with weak credentials discovered during the reconnaissance phase. This account provides access to a jump box within the cloud environment. From the jump box, the attacker utilizes a domain account (T1087.002) obtained through credential dumping to escalate privileges and gain access to sensitive cloud services like AWS S3 buckets containing operational data and system backups. The attacker specifically targets S3 buckets lacking proper access controls, allowing read and write access to unauthorized users.\\n\\nThe attacker then focuses on manipulating the Industrial Control System (ICS) environment. They identify and exploit vulnerabilities in the Human Machine Interfaces (HMIs) connected to the power grid, leveraging the compromised cloud infrastructure to deploy malicious code. This code is designed to disrupt power distribution by altering setpoints and causing cascading failures. Simultaneously, the attacker begins exfiltrating sensitive data, including customer information and operational schematics, to an external command-and-control server.\\n\\nThe attack is further complicated by the attacker’s attempts to compromise the security tooling and logging infrastructure. They attempt to disable or tamper with security information and event management (SIEM) systems to evade detection and cover their tracks. This is achieved by exploiting misconfigured logging policies and weak authentication mechanisms. The ultimate goal is to cause significant disruption to the power grid while simultaneously stealing valuable data for potential future exploitation or extortion.\\n\\nThe attacker’s actions result in a localized power outage affecting a critical infrastructure segment, alongside the theft of sensitive data. The incident highlights the importance of robust access controls, strong authentication mechanisms, and continuous monitoring of cloud environments, particularly within critical infrastructure sectors.\\n\\n[MATHEMATICAL EXPRESSION]\\nN = {n1: Employee Workstation, n2: Jump Box, n3: Domain Controller, n4: AWS S3 Bucket, n5: HMI, n6: SIEM System, n7: External C2 Server}\\nS = {s1: Weak Local Account Credentials, s2: Misconfigured IAM Roles, s3: Exposed Cloud Metadata Services, s4: Weak Authentication on SIEM, s5: Vulnerable HMI Software}\\nV = {v1: T1585.001 - Spearphishing, v2: T1027.013 - Encrypted/Encoded File, v3: T1136.001 - Local Account Exploitation, v4: T1018 - Remote System Discovery, v5: T1087.002 - Domain Account Exploitation, v6: Data Exfiltration, v7: ICS Manipulation}\\nP = {p1: n1 -> n2, p2: n2 -> n3, p3: n3 -> n4, p4: n2 -> n5, p5: n3 -> n6, p6: n4 -> n7, p7: n5 -> Disruption}\\nFS = {Localized Power Outage, Data Theft}\\n\\n[ATTACK GRAPH]\\nStage 1 (Initial Access): Employee Workstation [T1585.001] -> Execution of Encrypted File\\nStage 2 (Execution): Employee Workstation [T1027.013] -> PowerShell Script Execution\\nStage 3 (Persistence): Employee Workstation [T1136.001] -> Jump Box Access\\nStage 4 (Discovery): Jump Box [T1018] -> Internal Network Reconnaissance & Cloud Resource Identification\\nStage 5 (Lateral Movement): Jump Box [T1087.002] -> Domain Controller Access\\nStage 6 (Privilege Escalation): Domain Controller [T1087.002] -> AWS S3 Bucket Access\\nStage 7 (Data Exfiltration): AWS S3 Bucket [T1087.002] -> External C2 Server\\nStage 8 (ICS Manipulation): Jump Box -> HMI [T1087.002] -> Alter ICS Setpoints\\nStage 9 (Obfuscation): Domain Controller -> SIEM System -> Disable Logging\\nFinal Impact: Localized power outage and theft of sensitive operational data.\"}\n", encoding='utf-8')
pathlib.Path("requirements_finetune.txt").write_text("# Fine-tuning dependencies\n# Install with: pip install -r fine_tuning/requirements_finetune.txt\n#\n# Pinned to the stack that produced the shipped adapter (see output/README.md):\n#   Transformers 5.x, TRL 0.29.x, PEFT 0.18.x, datasets 4.x, torch 2.10, bnb 0.49.x.\n# The code branches on the TRL/transformers API differences (max_length vs\n# max_seq_length, processing_class vs tokenizer, the completion-only collator),\n# so nearby versions also work; these ranges are the verified-compatible set.\n\ntorch>=2.4\ntransformers>=5.0,<6\ntrl>=0.29,<0.30\npeft>=0.18,<0.19\ndatasets>=4.0,<5\naccelerate>=1.1\nbitsandbytes>=0.47,<0.50\n", encoding='utf-8')
print('Wrote:', "prompt_format.py", "data_split.py", "run_inference.py", "test_inputs.jsonl", "requirements_finetune.txt")

In [ ]:
import os
from google.colab import files
os.makedirs('output', exist_ok=True)
print('Upload the adapter files (adapter_model.safetensors, adapter_config.json,\ntokenizer files, chat_template.jinja) — they will be moved into ./output')
up = files.upload()
for fn in up:
    os.replace(fn, os.path.join('output', fn))
print('output/:', os.listdir('output'))

In [ ]:
# Decodes only the new tokens; MAX_NEW_TOKENS=1280 (in the 1024-1536 range).
!python run_inference.py --adapter ./output --test_inputs ./test_inputs.jsonl --out ./inference_results.txt --samples 3
# Add --compare to also generate the base (no-adapter) output side by side.

In [ ]:
print(open('inference_results.txt', encoding='utf-8').read())